# 0. Overview

This notebook analyzes whether the ENSO-850 hPa wind relationship over the Maritime Continent changed between **Old (1981-2006)** and **New (2007-2020)** periods, compared against the **Full (1981-2020)** baseline.

Scientific objective:
- Separate mean-state change (climatology) from teleconnection change (correlation/regression vs Niño3.4) and event-conditioned responses (El Niño / La Niña) in the 850 hPa wind field.
- Plot wind speed as shaded fields and overlay wind vectors with quiver arrows.


# 1. Import Library


In [ ]:
from pathlib import Path

# Define data paths (relative to notebook location)
RAINFALL_PATH = Path('../external/ClimateData/mswep-monthly/mswep_monthly_combined.nc').resolve()
WIND_PATH = Path('../external/ClimateData/era5-monthly/era5monthly_uvq_1980-2020.nc').resolve()
MFC_PATH = Path('../external/ClimateData/era5-monthly/vimf.nc').resolve()
SVP_PATH = Path('analysis_26-14_rev2/psi_chi_windparts_850.nc').resolve()
NINO34_PATH = Path('../external/ClimateData/index-monthly/nino34.anom.csv').resolve()

import numpy as np
import xarray as xr
import pandas as pd
import dask
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.pyplot as plt


In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'


# 2. Open Data & Pre-Process


#### Open ERA5 850 hPa Wind


In [ ]:
era5_path = wind_path = WIND_PATH
era5_chunks = {'valid_time': 12}

ds_era5 = xr.open_dataset(era5_path, chunks=era5_chunks)[['u', 'v']]
ds_era5 = ds_era5.sel(pressure_level=850)
ds_era5 = ds_era5.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
ds_era5 = ds_era5.assign_coords(lon=(ds_era5.lon % 360)).sortby('lon')
# slice wilayah maritime continent samudra pasifik dan samudra hindia
ds_era5 = ds_era5.sel(
    time=slice('1980-12-01', '2020-02-01'),
    lat=slice(21, -21),
    lon=slice(70, 290),
)
# keep DJF only, with December assigned to the following DJF year
month_mask = ds_era5.time.dt.month.isin([12, 1, 2])
djf_year = xr.where(ds_era5.time.dt.month == 12, ds_era5.time.dt.year + 1, ds_era5.time.dt.year)
ds_era5 = ds_era5.sel(time=month_mask).assign_coords(djf_year=('time', djf_year.sel(time=month_mask).data))

u850 = ds_era5['u']
v850 = ds_era5['v']
wind_speed = np.hypot(u850, v850)


#### Open NINO34 index


In [ ]:
nino34_path = nino34_path = NINO34_PATH
nino34_column = '   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/'

df_nino34 = pd.read_csv(
    nino34_path,
    usecols=['Date', nino34_column],
    parse_dates=['Date'],
)
df_nino34 = df_nino34.set_index('Date').loc['1980-12-01':'2020-02-29']
df_nino34 = df_nino34[df_nino34.index.month.isin([12, 1, 2])].copy()
df_nino34['djf_year'] = df_nino34.index.year + (df_nino34.index.month == 12).astype('int8')
df_nino34['DJF'] = 'DJF' + df_nino34['djf_year'].astype(str)


#### Define Period


In [ ]:
full_years = np.arange(1981, 2021)
past_years = np.arange(1981, 2007)
recent_years = np.arange(2007, 2021)

wind_clim = wind_speed.sel(time=wind_speed.djf_year.isin(full_years))
wind_past = wind_clim.sel(time=wind_clim.djf_year.isin(past_years))
wind_recent = wind_clim.sel(time=wind_clim.djf_year.isin(recent_years))

u_clim = u850.sel(time=u850.djf_year.isin(full_years))
u_past = u_clim.sel(time=u_clim.djf_year.isin(past_years))
u_recent = u_clim.sel(time=u_clim.djf_year.isin(recent_years))

v_clim = v850.sel(time=v850.djf_year.isin(full_years))
v_past = v_clim.sel(time=v_clim.djf_year.isin(past_years))
v_recent = v_clim.sel(time=v_clim.djf_year.isin(recent_years))

nino34_clim = df_nino34[df_nino34['djf_year'].isin(full_years)].copy()
nino34_past = nino34_clim[nino34_clim['djf_year'].isin(past_years)].copy()
nino34_recent = nino34_clim[nino34_clim['djf_year'].isin(recent_years)].copy()

period_coord = xr.DataArray(
    np.where(wind_clim.djf_year <= 2006, 'past', 'recent'),
    coords={'time': wind_clim.time},
    dims='time',
    name='period',
)
wind_period = wind_clim.assign_coords(period=period_coord)
u_period = u_clim.assign_coords(period=period_coord)
v_period = v_clim.assign_coords(period=period_coord)


#### Define Area


In [ ]:
# Define the extent and ticks 
# ip = Indonesia-Pacific region
ip_extent = [70, 290, -21, 21]
ip_yticks = [-15, 0, 15]
ip_xticks = np.arange(90, 291, 30)


In [ ]:
# Define the extent and ticks 
# mc: maritime continent
mc_extent = [90, 155, -15, 20]  # [lon_min, lon_max, lat_min, lat_max]
mc_yticks = np.arange(-10, 11, 10)
mc_xticks = np.arange(100, 151, 10)


# 3. Plot Climatology


In [ ]:
# Compute the shared climatology means once and reuse the period grouping
full_plot, period_means, full_u_plot, period_u_means, full_v_plot, period_v_means = dask.compute(
    wind_clim.mean('time'),
    wind_period.groupby('period').mean('time'),
    u_clim.mean('time'),
    u_period.groupby('period').mean('time'),
    v_clim.mean('time'),
    v_period.groupby('period').mean('time'),
)

# Transpose after loading (faster on in-memory numpy arrays)
full_plot = full_plot.transpose('lat', 'lon')
past_plot = period_means.sel(period='past').transpose('lat', 'lon')
recent_plot = period_means.sel(period='recent').transpose('lat', 'lon')
diff_plot = recent_plot - past_plot

full_u_plot = full_u_plot.transpose('lat', 'lon')
past_u_plot = period_u_means.sel(period='past').transpose('lat', 'lon')
recent_u_plot = period_u_means.sel(period='recent').transpose('lat', 'lon')
diff_u_plot = recent_u_plot - past_u_plot

full_v_plot = full_v_plot.transpose('lat', 'lon')
past_v_plot = period_v_means.sel(period='past').transpose('lat', 'lon')
recent_v_plot = period_v_means.sel(period='recent').transpose('lat', 'lon')
diff_v_plot = recent_v_plot - past_v_plot


In [ ]:


def _add_quiver(ax, u, v, step=16, scale=70, width=0.002, color='k', key_u=10, key_y=1.1, key_label=None, key_color='k'):
    u_plot = u.isel(lat=slice(None, None, step), lon=slice(None, None, step))
    v_plot = v.isel(lat=slice(None, None, step), lon=slice(None, None, step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=width,
        color=color,
        zorder=4,
    )
    if key_label is None:
        key_label = f'{key_u} m/s'
    ax.quiverkey(q, X=0.88, Y=key_y, U=key_u, label=key_label, labelpos='E', coordinates='axes', color=key_color)
    return q


def add_quiver_clim(ax, u, v, step=16, scale=70, width=0.002, color='k', key_y=1.1, key_u=10, key_label=None):
    return _add_quiver(ax, u, v, step=step, scale=scale, width=width, color=color, key_u=key_u, key_y=key_y, key_label=key_label)


def add_quiver_anom(ax, u, v, step=16, scale=70, width=0.002, key_u=2, color='gray', key_color='black', key_label=None, key_y=1.1):
    return _add_quiver(ax, u, v, step=step, scale=scale, width=width, color=color, 
                       key_u=key_u, key_y=key_y, key_label=key_label, key_color=key_color)


### Indo Pasifik


In [ ]:
plots = [
    (full_plot, full_u_plot, full_v_plot, '(a) DJF 1981-2020', 'rainbow', np.arange(0, 10.5, 0.5), np.arange(0, 11, 1), 'Kecepatan angin (m/s)', 'max'),
    (past_plot, past_u_plot, past_v_plot, '(b) DJF 1981-2006', 'rainbow', np.arange(0, 10.5, 0.5), np.arange(0, 11, 1), 'Kecepatan angin (m/s)', 'max'),
    (recent_plot, recent_u_plot, recent_v_plot, '(c) DJF 2007-2020', 'rainbow', np.arange(0, 10.5, 0.5), np.arange(0, 11, 1), 'Kecepatan angin (m/s)', 'max'),
    (diff_plot, diff_u_plot, diff_v_plot, '(d) Selisih (c)-(b)', 'RdYlGn_r', np.arange(-2, 2.1, 0.2), np.arange(-2, 2.1, 1), 'Selisih (m/s)', 'both'),
]

for data, u_data, v_data, title, cmap, levels, ticks, cbar_label, extend in plots:
    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    img = data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        add_labels=False,
        infer_intervals=True,
    )
    # step: distance between arrows larger means fewer arrows,
    # scale: adjust arrow length larger means shorter arrows, 
    # width: arrow thickness
    if data is diff_plot:
        add_quiver_anom(ax, u_data, v_data, step=20, scale=40, width=0.0017, color='black', 
                        key_u=1, key_label='1 m/s', key_color='black')
    else:
        add_quiver_clim(ax, u_data, v_data, step=23, scale=350, width=0.0017)

    ax.coastlines(resolution='10m', linewidth=0.5, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


## Maritime Continent


In [ ]:
for data, u_data, v_data, title, cmap, levels, ticks, cbar_label, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    img = data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        add_labels=False,
        infer_intervals=True,
    )

    # step: distance between arrows larger means fewer arrows,
    # scale: adjust arrow length larger means shorter arrows, 
    # width: arrow thickness
    if data is diff_plot:
        add_quiver_anom(ax, u_data, v_data, step=10, scale=38, width=0.002, color='black', 
                        key_u=1, key_label='1 m/s', key_color='black', key_y=1.05)
    else:
        add_quiver_clim(ax, u_data, v_data, step=10, scale=280, width=0.002, key_y=1.05)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


# 4. Plot Composite


### A. Analisis El Nino


In [ ]:
# define el nino years based on DJF-mean nino34 index threshold
elnino_threshold = 0.5
nino34_clim_djf = nino34_clim.groupby('djf_year')[nino34_column].mean().reset_index()
nino34_past_djf = nino34_past.groupby('djf_year')[nino34_column].mean().reset_index()
nino34_recent_djf = nino34_recent.groupby('djf_year')[nino34_column].mean().reset_index()

nino34_clim_elnino = nino34_clim_djf[nino34_clim_djf[nino34_column] >= elnino_threshold]
nino34_past_elnino = nino34_past_djf[nino34_past_djf[nino34_column] >= elnino_threshold]
nino34_recent_elnino = nino34_recent_djf[nino34_recent_djf[nino34_column] >= elnino_threshold]

elnino_years_clim = sorted(nino34_clim_elnino['djf_year'].tolist())
elnino_years_past = sorted(nino34_past_elnino['djf_year'].tolist())
elnino_years_recent = sorted(nino34_recent_elnino['djf_year'].tolist())
print('El Nino DJF years (1981-2020):', elnino_years_clim)
print('El Nino DJF years (1981-2006):', elnino_years_past)
print('El Nino DJF years (2007-2020):', elnino_years_recent)

wind_clim_elnino = wind_clim.sel(time=wind_clim.djf_year.isin(elnino_years_clim))
wind_past_elnino = wind_past.sel(time=wind_past.djf_year.isin(elnino_years_past))
wind_recent_elnino = wind_recent.sel(time=wind_recent.djf_year.isin(elnino_years_recent))

u_clim_elnino = u_clim.sel(time=u_clim.djf_year.isin(elnino_years_clim))
u_past_elnino = u_past.sel(time=u_past.djf_year.isin(elnino_years_past))
u_recent_elnino = u_recent.sel(time=u_recent.djf_year.isin(elnino_years_recent))

v_clim_elnino = v_clim.sel(time=v_clim.djf_year.isin(elnino_years_clim))
v_past_elnino = v_past.sel(time=v_past.djf_year.isin(elnino_years_past))
v_recent_elnino = v_recent.sel(time=v_recent.djf_year.isin(elnino_years_recent))

wind_clim_elnino, wind_past_elnino, wind_recent_elnino, u_clim_elnino, u_past_elnino, u_recent_elnino, v_clim_elnino, v_past_elnino, v_recent_elnino = dask.compute(
    wind_clim_elnino.mean('time'),
    wind_past_elnino.mean('time'),
    wind_recent_elnino.mean('time'),
    u_clim_elnino.mean('time'),
    u_past_elnino.mean('time'),
    u_recent_elnino.mean('time'),
    v_clim_elnino.mean('time'),
    v_past_elnino.mean('time'),
    v_recent_elnino.mean('time'),
)

wind_clim_elnino = wind_clim_elnino.transpose('lat', 'lon')
wind_past_elnino = wind_past_elnino.transpose('lat', 'lon')
wind_recent_elnino = wind_recent_elnino.transpose('lat', 'lon')

u_clim_elnino = u_clim_elnino.transpose('lat', 'lon')
u_past_elnino = u_past_elnino.transpose('lat', 'lon')
u_recent_elnino = u_recent_elnino.transpose('lat', 'lon')

v_clim_elnino = v_clim_elnino.transpose('lat', 'lon')
v_past_elnino = v_past_elnino.transpose('lat', 'lon')
v_recent_elnino = v_recent_elnino.transpose('lat', 'lon')

u_clim_elnino_anom = (u_clim_elnino - full_u_plot).reset_coords(drop=True)
u_past_elnino_anom = (u_past_elnino - past_u_plot).reset_coords(drop=True)
u_recent_elnino_anom = (u_recent_elnino - recent_u_plot).reset_coords(drop=True)

v_clim_elnino_anom = (v_clim_elnino - full_v_plot).reset_coords(drop=True)
v_past_elnino_anom = (v_past_elnino - past_v_plot).reset_coords(drop=True)
v_recent_elnino_anom = (v_recent_elnino - recent_v_plot).reset_coords(drop=True)

diff_u_elnino_anom = (u_recent_elnino_anom - u_past_elnino_anom).reset_coords(drop=True)
diff_v_elnino_anom = (v_recent_elnino_anom - v_past_elnino_anom).reset_coords(drop=True)

wind_clim_elnino_anom = wind_clim_elnino - full_plot
wind_past_elnino_anom = wind_past_elnino - past_plot
wind_recent_elnino_anom = wind_recent_elnino - recent_plot

wind_clim_elnino_anom = wind_clim_elnino_anom.reset_coords(drop=True)
wind_past_elnino_anom = wind_past_elnino_anom.reset_coords(drop=True)
wind_recent_elnino_anom = wind_recent_elnino_anom.reset_coords(drop=True)
diff_elnino_anom = wind_recent_elnino_anom - wind_past_elnino_anom

#### plot komposit indo pasifik


In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (wind_clim_elnino_anom, u_clim_elnino_anom, v_clim_elnino_anom, '(a) Komposit El Nino DJF (1981 - 2020)', 'Spectral', 'Anomali kecepatan angin (m/s)', np.arange(-5, 5.1, 0.5), np.arange(-5, 5.1, 1), 'both'),
    (wind_past_elnino_anom, u_past_elnino_anom, v_past_elnino_anom, '(b) Komposit El Nino DJF (1981 - 2006)', 'Spectral', 'Anomali kecepatan angin (m/s)', np.arange(-5, 5.1, 0.5), np.arange(-5, 5.1, 1), 'both'),
    (wind_recent_elnino_anom, u_recent_elnino_anom, v_recent_elnino_anom, '(c) Komposit El Nino DJF (2007 - 2020)', 'Spectral', 'Anomali kecepatan angin (m/s)', np.arange(-5, 5.1, 0.5), np.arange(-5, 5.1, 1), 'both'),
    (diff_elnino_anom, diff_u_elnino_anom, diff_v_elnino_anom, '(d) Selisih (c) - (b)', 'Spectral_r', 'Selisih (m/s)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
]

for data, u_data, v_data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )
    
    if data is diff_elnino_anom:
        add_quiver_anom(ax, u_data, v_data, step=20, scale=50, width=0.0017, color='black', 
                        key_u=1, key_label='1 m/s', key_color='black')
    else:
        add_quiver_clim(ax, u_data, v_data, step=23, scale=50, width=0.0017, key_u=1, key_label='1 m/s', key_y=1.05)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (rain_clim_lanina_anom, '(a) Komposit La Nina DJF (1981 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_past_lanina_anom, '(b) Komposit La Nina DJF (1981 - 2006)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_recent_lanina_anom, '(c) Komposit La Nina DJF (2007 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    ((rain_recent_lanina_anom - rain_past_lanina_anom), '(d) Selisih (c) - (b)', 'BrBG', 'Selisih (mm/bulan)', np.arange(-100, 101, 5), np.arange(-100, 101, 25), 'both'),
]


In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (rain_clim_lanina_anom, '(a) Komposit La Nina DJF (1981 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_past_lanina_anom, '(b) Komposit La Nina DJF (1981 - 2006)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_recent_lanina_anom, '(c) Komposit La Nina DJF (2007 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    ((rain_recent_lanina_anom - rain_past_lanina_anom), '(d) Selisih (c) - (b)', 'BrBG', 'Selisih (mm/bulan)', np.arange(-100, 101, 5), np.arange(-100, 101, 25), 'both'),
]


#### plot mc


In [ ]:

for data, u_data, v_data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    if data is diff_elnino_anom:
        add_quiver_anom(ax, u_data, v_data, step=10, scale=50, width=0.0017, color='black', 
                        key_u=1, key_label='1 m/s', key_color='black')
    else:
        add_quiver_clim(ax, u_data, v_data, step=13, scale=50, width=0.0017, key_u=1, key_label='1 m/s', key_y=1.05)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (rain_clim_lanina_anom, '(a) Komposit La Nina DJF (1981 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_past_lanina_anom, '(b) Komposit La Nina DJF (1981 - 2006)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_recent_lanina_anom, '(c) Komposit La Nina DJF (2007 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    ((rain_recent_lanina_anom - rain_past_lanina_anom), '(d) Selisih (c) - (b)', 'BrBG', 'Selisih (mm/bulan)', np.arange(-100, 101, 5), np.arange(-100, 101, 25), 'both'),
]


### B. Analisis La Nina


In [ ]:
# define la nina years based on DJF-mean nino34 index threshold
lanina_threshold = -0.5

nino34_clim_lanina = nino34_clim_djf[nino34_clim_djf[nino34_column] <= lanina_threshold]
nino34_past_lanina = nino34_past_djf[nino34_past_djf[nino34_column] <= lanina_threshold]
nino34_recent_lanina = nino34_recent_djf[nino34_recent_djf[nino34_column] <= lanina_threshold]

lanina_years_clim = sorted(nino34_clim_lanina['djf_year'].tolist())
lanina_years_past = sorted(nino34_past_lanina['djf_year'].tolist())
lanina_years_recent = sorted(nino34_recent_lanina['djf_year'].tolist())
print('La Nina DJF years (1981-2020):', lanina_years_clim)
print('La Nina DJF years (1981-2006):', lanina_years_past)
print('La Nina DJF years (2007-2020):', lanina_years_recent)

wind_clim_lanina = wind_clim.sel(time=wind_clim.djf_year.isin(lanina_years_clim))
wind_past_lanina = wind_past.sel(time=wind_past.djf_year.isin(lanina_years_past))
wind_recent_lanina = wind_recent.sel(time=wind_recent.djf_year.isin(lanina_years_recent))

u_clim_lanina = u_clim.sel(time=u_clim.djf_year.isin(lanina_years_clim))
u_past_lanina = u_past.sel(time=u_past.djf_year.isin(lanina_years_past))
u_recent_lanina = u_recent.sel(time=u_recent.djf_year.isin(lanina_years_recent))

v_clim_lanina = v_clim.sel(time=v_clim.djf_year.isin(lanina_years_clim))
v_past_lanina = v_past.sel(time=v_past.djf_year.isin(lanina_years_past))
v_recent_lanina = v_recent.sel(time=v_recent.djf_year.isin(lanina_years_recent))

wind_clim_lanina, wind_past_lanina, wind_recent_lanina, u_clim_lanina, u_past_lanina, u_recent_lanina, v_clim_lanina, v_past_lanina, v_recent_lanina = dask.compute(
    wind_clim_lanina.mean('time'),
    wind_past_lanina.mean('time'),
    wind_recent_lanina.mean('time'),
    u_clim_lanina.mean('time'),
    u_past_lanina.mean('time'),
    u_recent_lanina.mean('time'),
    v_clim_lanina.mean('time'),
    v_past_lanina.mean('time'),
    v_recent_lanina.mean('time'),
)

wind_clim_lanina = wind_clim_lanina.transpose('lat', 'lon')
wind_past_lanina = wind_past_lanina.transpose('lat', 'lon')
wind_recent_lanina = wind_recent_lanina.transpose('lat', 'lon')

u_clim_lanina = u_clim_lanina.transpose('lat', 'lon')
u_past_lanina = u_past_lanina.transpose('lat', 'lon')
u_recent_lanina = u_recent_lanina.transpose('lat', 'lon')

v_clim_lanina = v_clim_lanina.transpose('lat', 'lon')
v_past_lanina = v_past_lanina.transpose('lat', 'lon')
v_recent_lanina = v_recent_lanina.transpose('lat', 'lon')

u_clim_lanina_anom = (u_clim_lanina - full_u_plot).reset_coords(drop=True)
u_past_lanina_anom = (u_past_lanina - past_u_plot).reset_coords(drop=True)
u_recent_lanina_anom = (u_recent_lanina - recent_u_plot).reset_coords(drop=True)

v_clim_lanina_anom = (v_clim_lanina - full_v_plot).reset_coords(drop=True)
v_past_lanina_anom = (v_past_lanina - past_v_plot).reset_coords(drop=True)
v_recent_lanina_anom = (v_recent_lanina - recent_v_plot).reset_coords(drop=True)

diff_u_lanina_anom = (u_recent_lanina_anom - u_past_lanina_anom).reset_coords(drop=True)
diff_v_lanina_anom = (v_recent_lanina_anom - v_past_lanina_anom).reset_coords(drop=True)

wind_clim_lanina_anom = wind_clim_lanina - full_plot
wind_past_lanina_anom = wind_past_lanina - past_plot
wind_recent_lanina_anom = wind_recent_lanina - recent_plot
diff_lanina_anom = wind_recent_lanina_anom - wind_past_lanina_anom

wind_clim_lanina_anom = wind_clim_lanina_anom.reset_coords(drop=True)
wind_past_lanina_anom = wind_past_lanina_anom.reset_coords(drop=True)
wind_recent_lanina_anom = wind_recent_lanina_anom.reset_coords(drop=True)


In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (wind_clim_lanina_anom, u_clim_lanina_anom, v_clim_lanina_anom, '(a) Komposit La Nina DJF (1981 - 2020)', 'Spectral', 'Anomali kecepatan angin (m/s)', np.arange(-5, 5.1, 0.5), np.arange(-5, 5.1, 1), 'both'),
    (wind_past_lanina_anom, u_past_lanina_anom, v_past_lanina_anom, '(b) Komposit La Nina DJF (1981 - 2006)', 'Spectral', 'Anomali kecepatan angin (m/s)', np.arange(-5, 5.1, 0.5), np.arange(-5, 5.1, 1), 'both'),
    (wind_recent_lanina_anom, u_recent_lanina_anom, v_recent_lanina_anom, '(c) Komposit La Nina DJF (2007 - 2020)', 'Spectral', 'Anomali kecepatan angin (m/s)', np.arange(-5, 5.1, 0.5), np.arange(-5, 5.1, 1), 'both'),
    (diff_lanina_anom, diff_u_lanina_anom, diff_v_lanina_anom, '(d) Selisih (c) - (b)', 'Spectral_r', 'Selisih (m/s)', np.arange(-4, 4.1, 0.25), np.arange(-4, 4.1, 1), 'both'),
]


In [ ]:
for data, u_data, v_data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    if data is diff_lanina_anom:
        add_quiver_anom(ax, u_data, v_data, step=20, scale=50, width=0.0017, color='black', 
                        key_u=1, key_label='1 m/s', key_color='black')
    else:
        add_quiver_clim(ax, u_data, v_data, step=23, scale=50, width=0.0017, key_u=1, key_label='1 m/s', key_y=1.05)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


In [ ]:
#data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (rain_clim_lanina_anom, '(a) Komposit La Nina DJF (1981 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_past_lanina_anom, '(b) Komposit La Nina DJF (1981 - 2006)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    (rain_recent_lanina_anom, '(c) Komposit La Nina DJF (2007 - 2020)', 'RdBu', 'Anomali curah hujan (mm/bulan)', np.arange(-150, 151, 10), np.arange(-150, 151, 50), 'both'),
    ((rain_recent_lanina_anom - rain_past_lanina_anom), '(d) Selisih (c) - (b)', 'BrBG', 'Selisih (mm/bulan)', np.arange(-100, 101, 5), np.arange(-100, 101, 25), 'both'),
]


In [ ]:
for data, u_data, v_data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    if data is diff_lanina_anom:
        add_quiver_anom(ax, u_data, v_data, step=10, scale=50, width=0.0017, color='black', 
                        key_u=1, key_label='1 m/s', key_color='black')
    else:
        add_quiver_clim(ax, u_data, v_data, step=13, scale=50, width=0.0017, key_u=1, key_label='1 m/s', key_y=1.05)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


# 5. Plot Correlation


Compute the DJF teleconnection as Pearson correlation between DJF-mean u850/v850 and DJF-mean Niño3.4, then plot quiver-only correlation vectors for Indo-Pacific and Maritime Continent.

Requirements:
- Use one DJF seasonal mean wind value per djf_year
- Use one DJF seasonal mean Niño3.4 value per djf_year
- Keep the DJF year convention as already defined
- Compute correlation separately for u850 and v850 with xr.corr(..., dim='djf_year')
- Plot the four periods: full, past, recent, and recent minus past
- Use quiver only: no shading, no pcolormesh, no colorbar, no stippling, no significance test
- Reuse the existing domain, projection, tick, coastline, and map style
- Add a quiver key


In [ ]:
# Compute DJF-mean u850/v850 and DJF-mean Niño3.4, then correlate by djf_year
u_clim_djf_corr = u_clim.groupby('djf_year').mean('time')
u_past_djf_corr = u_past.groupby('djf_year').mean('time')
u_recent_djf_corr = u_recent.groupby('djf_year').mean('time')

v_clim_djf_corr = v_clim.groupby('djf_year').mean('time')
v_past_djf_corr = v_past.groupby('djf_year').mean('time')
v_recent_djf_corr = v_recent.groupby('djf_year').mean('time')

nino34_clim_djf_corr_series = nino34_clim.groupby('djf_year')[nino34_column].mean()
nino34_past_djf_corr_series = nino34_past.groupby('djf_year')[nino34_column].mean()
nino34_recent_djf_corr_series = nino34_recent.groupby('djf_year')[nino34_column].mean()

nino34_clim_djf_corr = xr.DataArray(
    nino34_clim_djf_corr_series.to_numpy(),
    coords={'djf_year': nino34_clim_djf_corr_series.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)
nino34_past_djf_corr = xr.DataArray(
    nino34_past_djf_corr_series.to_numpy(),
    coords={'djf_year': nino34_past_djf_corr_series.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)
nino34_recent_djf_corr = xr.DataArray(
    nino34_recent_djf_corr_series.to_numpy(),
    coords={'djf_year': nino34_recent_djf_corr_series.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)

u_clim_djf_corr, nino34_clim_djf_corr = xr.align(u_clim_djf_corr, nino34_clim_djf_corr, join='inner')
u_past_djf_corr, nino34_past_djf_corr = xr.align(u_past_djf_corr, nino34_past_djf_corr, join='inner')
u_recent_djf_corr, nino34_recent_djf_corr = xr.align(u_recent_djf_corr, nino34_recent_djf_corr, join='inner')

v_clim_djf_corr, _ = xr.align(v_clim_djf_corr, nino34_clim_djf_corr, join='inner')
v_past_djf_corr, _ = xr.align(v_past_djf_corr, nino34_past_djf_corr, join='inner')
v_recent_djf_corr, _ = xr.align(v_recent_djf_corr, nino34_recent_djf_corr, join='inner')

corr_u_clim, corr_u_past, corr_u_recent = dask.compute(
    xr.corr(u_clim_djf_corr, nino34_clim_djf_corr, dim='djf_year'),
    xr.corr(u_past_djf_corr, nino34_past_djf_corr, dim='djf_year'),
    xr.corr(u_recent_djf_corr, nino34_recent_djf_corr, dim='djf_year'),
)
corr_v_clim, corr_v_past, corr_v_recent = dask.compute(
    xr.corr(v_clim_djf_corr, nino34_clim_djf_corr, dim='djf_year'),
    xr.corr(v_past_djf_corr, nino34_past_djf_corr, dim='djf_year'),
    xr.corr(v_recent_djf_corr, nino34_recent_djf_corr, dim='djf_year'),
)

corr_u_recent_minus_past = corr_u_recent - corr_u_past
corr_v_recent_minus_past = corr_v_recent - corr_v_past

print('Sample size full:', int(u_clim_djf_corr.sizes['djf_year']))
print('Sample size past:', int(u_past_djf_corr.sizes['djf_year']))
print('Sample size recent:', int(u_recent_djf_corr.sizes['djf_year']))


In [ ]:
# No significance testing is applied for the correlation quiver plots.


In [ ]:
# plot indo pasifik
corr_quiver_key_u = 1.0
corr_quiver_key_label = 'corr = 1.0'
corr_quiver_scale = 30
corr_step_ip = 20
corr_step_mc = 8

corr_diff_quiver_key_u = 0.5
corr_diff_quiver_key_label = 'corr = 0.5'
corr_diff_quiver_scale = 15
corr_diff_step_ip = 20
corr_diff_step_mc = 8

plot_defs = [
    (corr_u_clim, corr_v_clim, '(a) Korelasi Angin (u,v) vs Niño3.4 DJF 1981-2020', False),
    (corr_u_past, corr_v_past, '(b) Korelasi Angin (u,v) vs Niño3.4 DJF 1981-2006', False),
    (corr_u_recent, corr_v_recent, '(c) Korelasi Angin (u,v) vs Niño3.4 DJF 2007-2020', False),
    (corr_u_recent_minus_past, corr_v_recent_minus_past, '(d) Korelasi Angin (u,v) vs Niño3.4 DJF Selisih (c) - (b)', True),
]

for u_data, v_data, title, is_diff in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    if is_diff:
        add_quiver_anom(
            ax,
            u_data,
            v_data,
            step=corr_diff_step_ip,
            scale=corr_diff_quiver_scale,
            width=0.002,
            color='black',
            key_u=corr_diff_quiver_key_u,
            key_label=corr_diff_quiver_key_label,
            key_color='black',
            key_y=1.06,
        )
    else:
        add_quiver_clim(
            ax,
            u_data,
            v_data,
            step=corr_step_ip,
            scale=corr_quiver_scale,
            width=0.002,
            color='black',
            key_u=corr_quiver_key_u,
            key_label=corr_quiver_key_label,
            key_y=1.06,
        )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    plt.show()


In [ ]:
# plot maritime continent

plot_defs = [
    (corr_u_clim, corr_v_clim, '(a) Korelasi Angin (u,v) vs Niño3.4 DJF 1981-2020', False),
    (corr_u_past, corr_v_past, '(b) Korelasi Angin (u,v) vs Niño3.4 DJF 1981-2006', False),
    (corr_u_recent, corr_v_recent, '(c) Korelasi Angin (u,v) vs Niño3.4 DJF 2007-2020', False),
    (corr_u_recent_minus_past, corr_v_recent_minus_past, '(d) Korelasi Angin (u,v) vs Niño3.4 DJF Selisih (c) - (b)', True),
]

for u_data, v_data, title, is_diff in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    if is_diff:
        add_quiver_anom(
            ax,
            u_data,
            v_data,
            step=corr_diff_step_mc,
            scale=corr_diff_quiver_scale,
            width=0.002,
            color='black',
            key_u=corr_diff_quiver_key_u,
            key_label=corr_diff_quiver_key_label,
            key_color='black',
            key_y=1.06,
        )
    else:
        add_quiver_clim(
            ax,
            u_data,
            v_data,
            step=corr_step_mc,
            scale=corr_quiver_scale,
            width=0.002,
            color='black',
            key_u=corr_quiver_key_u,
            key_label=corr_quiver_key_label,
            key_y=1.06,
        )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    plt.show()


# 6. Plot Regresi


In [ ]:
# Reuse the DJF-mean wind speed and DJF-mean Niño3.4 from the correlation section, then regress wind speed on Niño3.4 by djf_year
from scipy.stats import linregress

def _linregress_1d(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    result = linregress(x[mask], y[mask])
    return result.slope, result.intercept, result.rvalue, result.pvalue, result.stderr

wind_clim_djf_reg, nino34_clim_djf_reg = xr.align(wind_clim_djf_corr, nino34_clim_djf_corr, join='inner')
wind_past_djf_reg, nino34_past_djf_reg = xr.align(wind_past_djf_corr, nino34_past_djf_corr, join='inner')
wind_recent_djf_reg, nino34_recent_djf_reg = xr.align(wind_recent_djf_corr, nino34_recent_djf_corr, join='inner')

wind_clim_djf_reg = wind_clim_djf_reg.chunk({'djf_year': -1})
nino34_clim_djf_reg = nino34_clim_djf_reg.chunk({'djf_year': -1})
wind_past_djf_reg = wind_past_djf_reg.chunk({'djf_year': -1})
nino34_past_djf_reg = nino34_past_djf_reg.chunk({'djf_year': -1})
wind_recent_djf_reg = wind_recent_djf_reg.chunk({'djf_year': -1})
nino34_recent_djf_reg = nino34_recent_djf_reg.chunk({'djf_year': -1})

reg_clim = xr.apply_ufunc(
    _linregress_1d,
    nino34_clim_djf_reg,
    wind_clim_djf_reg,
    input_core_dims=[['djf_year'], ['djf_year']],
    output_core_dims=[[], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float, float, float, float, float],
)
reg_past = xr.apply_ufunc(
    _linregress_1d,
    nino34_past_djf_reg,
    wind_past_djf_reg,
    input_core_dims=[['djf_year'], ['djf_year']],
    output_core_dims=[[], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float, float, float, float, float],
)
reg_recent = xr.apply_ufunc(
    _linregress_1d,
    nino34_recent_djf_reg,
    wind_recent_djf_reg,
    input_core_dims=[['djf_year'], ['djf_year']],
    output_core_dims=[[], [], [], [], []],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float, float, float, float, float],
)

reg_clim_slope, reg_clim_intercept, reg_clim_r, reg_clim_p, reg_clim_stderr = dask.compute(*reg_clim)
reg_past_slope, reg_past_intercept, reg_past_r, reg_past_p, reg_past_stderr = dask.compute(*reg_past)
reg_recent_slope, reg_recent_intercept, reg_recent_r, reg_recent_p, reg_recent_stderr = dask.compute(*reg_recent)
reg_recent_minus_past_slope = reg_recent_slope - reg_past_slope

reg_clim_sig = reg_clim_p < 0.05
reg_past_sig = reg_past_p < 0.05
reg_recent_sig = reg_recent_p < 0.05

reg_absmax = xr.concat([
    np.abs(reg_clim_slope),
    np.abs(reg_past_slope),
    np.abs(reg_recent_slope),
    np.abs(reg_recent_minus_past_slope),
], dim='stack').max()
reg_absmax = float(reg_absmax)
reg_absmax = max(reg_absmax, 0.05)
reg_absmax = np.ceil(reg_absmax / 0.05) * 0.05
reg_levels = np.linspace(-reg_absmax, reg_absmax, 21)
reg_ticks = np.linspace(-reg_absmax, reg_absmax, 9)

print('Sample size full:', int(wind_clim_djf_reg.sizes['djf_year']))
print('Sample size past:', int(wind_past_djf_reg.sizes['djf_year']))
print('Sample size recent:', int(wind_recent_djf_reg.sizes['djf_year']))


In [ ]:
reg_levels = np.arange(-4, 4.1, 0.25)
reg_ticks = np.arange(-4, 4.1, 1)

In [ ]:
# plot regression indo pacific

plot_defs = [
    (reg_clim_slope, reg_clim_sig, '(a) Regresi Kecepatan Angin vs Niño3.4 DJF 1981-2020', 'PiYG', 'Kemiringan regresi kecepatan angin (m/s per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_past_slope, reg_past_sig, '(b) Regresi Kecepatan Angin vs Niño3.4 DJF 1981-2006', 'PiYG', 'Kemiringan regresi kecepatan angin (m/s per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_recent_slope, reg_recent_sig, '(c) Regresi Kecepatan Angin vs Niño3.4 DJF 2007-2020', 'PiYG', 'Kemiringan regresi kecepatan angin (m/s per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_recent_minus_past_slope, None, '(d) Selisih kemiringan (c) - (b)', 'PiYG_r', 'Selisih kemiringan regresi kecepatan angin (m/s per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
]

for data, sig_mask, title, cmap, cbar_label, levels, ticks, extend in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    if sig_mask is not None:
        add_stippling(ax, sig_mask, block=20, size=15, alpha=0.9)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


In [ ]:
# plot regression maritime continent

plot_defs = [
    (reg_clim_slope, reg_clim_sig, '(a) Regresi Kecepatan Angin vs Niño3.4 DJF 1981-2020', 'PiYG', 'Kemiringan regresi kecepatan angin (m/s per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_past_slope, reg_past_sig, '(b) Regresi Kecepatan Angin vs Niño3.4 DJF 1981-2006', 'PiYG', 'Kemiringan regresi kecepatan angin (m/s per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_recent_slope, reg_recent_sig, '(c) Regresi Kecepatan Angin vs Niño3.4 DJF 2007-2020', 'PiYG', 'Kemiringan regresi kecepatan angin (m/s per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
    (reg_recent_minus_past_slope, None, '(d) Selisih kemiringan (c) - (b)', 'PiYG_r', 'Selisih kemiringan regresi kecepatan angin (m/s per 1°C anomali Niño3.4)', reg_levels, reg_ticks, 'both'),
]

for data, sig_mask, title, cmap, cbar_label, levels, ticks, extend in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    if sig_mask is not None:
        add_stippling(ax, sig_mask, block=7, size=15, alpha=0.9)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()
